# ej in 5 minutes

[ej](https://github.com/5ak3t/ej) is an 11 MB model that makes typed decisions on CPU, with no LLM. This notebook installs it,
routes support tickets, adapts it to your labels, and picks a confidence threshold for automation. It runs on the free CPU runtime.

In [ ]:
!pip install -q ejai

In [ ]:
import ej

model = ej.load('5ak3t/ej', revision='v0.0.1')   # downloads one 11 MB file

TEAMS = [('billing', 'Billing and payments'), ('returns', 'Returns and replacements'),
         ('tech', 'Technical problems'), ('account', 'Account and login')]
QUESTION = {'type': 'choice', 'instructions': 'Which team should handle this ticket?',
            'options': [{'key': k, 'text': t} for k, t in TEAMS]}
KEYS = [k for k, _ in TEAMS]

def record(text, label=None):
    r = {'state': text, 'questions': {'team': QUESTION}}
    if label:
        r['answers'] = {'team': label}
    return r

(probs,) = model.predict([record('You charged my card twice this month.')])
dict(zip(KEYS, [round(p, 3) for p in probs['team']]))

## Adapt to your workflow

`adapt` takes a few labelled records and learns per-option offsets; the weights do not change. On ej 0.0.1 it mostly teaches
the label prior, so expect small gains. Measure on records you did not adapt on.

In [ ]:
labelled = [('You charged my card twice.', 'billing'), ('Where is my refund for last month?', 'billing'),
            ('The jug arrived cracked, please send a new one.', 'returns'), ('Wrong size shoes, I want to swap them.', 'returns'),
            ('The app freezes on the checkout page.', 'tech'), ('Push notifications stopped after the update.', 'tech'),
            ('I forgot my password and the reset email never arrives.', 'account'), ('How do I change my account email?', 'account')]
held_out = [('My invoice shows the wrong VAT number.', 'billing'), ('The blender lid is missing, can I get a replacement?', 'returns'),
            ('The website shows a 500 error when I log in.', 'tech'), ('Please delete my account and all my data.', 'account'),
            ('I was billed after I cancelled.', 'billing'), ('The screen crashes when I upload a photo.', 'tech'),
            ('The package arrived damaged.', 'returns'), ('I cannot sign in with Google anymore.', 'account')]

adapted = model.adapt(examples=[record(t, y) for t, y in labelled])

def score(m):
    out = m.predict([record(t) for t, _ in held_out])
    rows = []
    for p, (t, y) in zip(out, held_out):
        best = max(range(len(KEYS)), key=lambda i: p['team'][i])
        rows.append((KEYS[best], p['team'][best], y))
    return rows

for name, m in (('zero-shot', model), ('adapted', adapted)):
    rows = score(m)
    print(f"{name:9}  accuracy {sum(p == y for p, _, y in rows) / len(rows):.2f}")

## Pick a threshold for automation

Route a ticket automatically only when ej is confident; send the rest to a person. Choose the threshold on labelled records of
**your** workflow: a threshold chosen on one workflow does not carry over to another. Eight records are only enough to show the
idea; use a few hundred in practice.

In [ ]:
rows = score(adapted)
print('threshold  automated  accuracy on automated')
for t in (0.5, 0.6, 0.7, 0.8, 0.9, 0.95):
    auto = [(p, y) for p, c, y in rows if c >= t]
    acc = sum(p == y for p, y in auto) / len(auto) if auto else float('nan')
    print(f'{t:9.2f}  {len(auto):4d}/{len(rows)}  {acc:10.2f}')

## Next

- Write your own options: any labels, chosen at run time.
- Question types: `choice`, `noul` (yes/no) and `score` (ordered levels). In 0.0.1, `choice` is the strongest.
- [README](https://github.com/5ak3t/ej#readme) · [What to expect](https://github.com/5ak3t/ej#what-to-expect)